In [1]:
# Parameters
DB_PATH          = "../../../DB/oedb_baseline_v3.db"
BENCHMARK_PATH   = "../../../data/input_data/benchmark_trainingset.xlsx"
BENCHMARK_SHEET  = "notegroups"
NOTEGROUP_ID_MIN = 1
NOTEGROUP_ID_MAX = 23
FIELDS           = ["date", "data_source_category"]

In [2]:
import sqlite3
import pandas as pd
from datetime import datetime

def load_etl(db_path, id_min, id_max):
    con = sqlite3.connect(db_path)
    df = pd.read_sql_query(
        "SELECT notegroupID, date, data_source_category FROM notegroups WHERE notegroupID BETWEEN ? AND ?",
        con, params=(id_min, id_max)
    )
    con.close()
    df["notegroupID"] = df["notegroupID"].astype(int)
    return df.set_index("notegroupID")

def load_benchmark(xlsx_path, sheet, id_min, id_max):
    df = pd.read_excel(xlsx_path, sheet_name=sheet, dtype=str)
    df["notegroupID"] = df["notegroupID"].astype(int)
    df = df[df["notegroupID"].between(id_min, id_max)]
    return df.set_index("notegroupID")[FIELDS]

etl = load_etl(DB_PATH, NOTEGROUP_ID_MIN, NOTEGROUP_ID_MAX)
bm  = load_benchmark(BENCHMARK_PATH, BENCHMARK_SHEET, NOTEGROUP_ID_MIN, NOTEGROUP_ID_MAX)

print("ETL records:      ", len(etl))
print("Benchmark records:", len(bm))

ETL records:       23
Benchmark records: 23


In [3]:
matched_ids = sorted(set(etl.index) & set(bm.index))
etl_only    = sorted(set(etl.index) - set(bm.index))
bm_only     = sorted(set(bm.index)  - set(etl.index))

print(f"Matched pairs : {len(matched_ids)}")
print(f"ETL-only (→ FP rows) : {etl_only}")
print(f"BM-only  (→ FN rows) : {bm_only}")

Matched pairs : 23
ETL-only (→ FP rows) : []
BM-only  (→ FN rows) : []


In [4]:
def normalise(val, is_date=False):
    if pd.isna(val) or str(val).strip() in ("", "None", "nan", "NaT"):
        return None
    s = str(val).strip()
    if is_date:
        s = s.split(" ")[0]  # drop time component if present
        for fmt in (
            "%Y-%m-%d",                  # after split: both sides land here
            "%b %d, %Y,",               # DB: Nov 22, 1111, 12:00:00 AM (pre-split)
            "%m/%d/%Y",                  # BM: 11/22/1111
        ):
            try:
                return datetime.strptime(s, fmt).strftime("%Y-%m-%d")
            except ValueError:
                continue
        return s.lower()
    return s.lower()

def compute_counts(etl_val, bm_val, is_date=False):
    """Return (TP, FP, FN, TN) for one field comparison."""
    e = normalise(etl_val, is_date)
    b = normalise(bm_val, is_date)
    if e is not None and b is not None:
        return (1, 0, 0, 0) if e == b else (0, 1, 1, 0)  #matched  → TP mismatch → FP + FN
    if e is not None and b is None:
        return (0, 1, 0, 0)   # hallucinated → FP
    if e is None and b is not None:
        return (0, 0, 1, 0)   # missed → FN
    return (0, 0, 0, 1)       # both null → TN

def safe_div(num, den):
    return round(num / den, 4) if den > 0 else None

def metrics_from_counts(TP, FP, FN, TN):
    accuracy  = safe_div(TP + TN, TP + FP + FN + TN)
    precision = safe_div(TP, TP + FP)
    recall    = safe_div(TP, TP + FN)
    f1 = round(2 * precision * recall / (precision + recall), 4) \
         if precision and recall and (precision + recall) > 0 else None
    return dict(TP=TP, FP=FP, FN=FN, TN=TN,
                accuracy=accuracy, precision=precision, recall=recall, F1=f1)

In [5]:
totals = {f: dict(TP=0, FP=0, FN=0, TN=0) for f in FIELDS}

# Matched pairs — field-level comparison
for nid in matched_ids:
    for field in FIELDS:
        tp, fp, fn, tn = compute_counts(
            etl.at[nid, field] if field in etl.columns else None,
            bm.at[nid,  field] if field in bm.columns  else None,
            is_date=(field == "date")
        )
        totals[field]["TP"] += tp; totals[field]["FP"] += fp
        totals[field]["FN"] += fn; totals[field]["TN"] += tn

# ETL-only rows → every field counts as FP (record shouldn't exist)
for nid in etl_only:
    for field in FIELDS:
        totals[field]["FP"] += 1

# BM-only rows → every field counts as FN (record was missed entirely)
for nid in bm_only:
    for field in FIELDS:
        totals[field]["FN"] += 1

In [6]:
# Per-field results table
rows = []
for field in FIELDS:
    m = metrics_from_counts(**totals[field])
    rows.append({"field": field, **m})

# Overall (aggregate across all fields)
overall = {k: sum(totals[f][k] for f in FIELDS) for k in ("TP","FP","FN","TN")}
m_all = metrics_from_counts(**overall)
rows.append({"field": "OVERALL", **m_all})

results_df = pd.DataFrame(rows).set_index("field")
results_df

,TP,FP,FN,TN,accuracy,precision,recall,F1
field,,,,,,,,
date,11,0,2,10,0.9130,1.0000,0.8462,0.9167
data_source_category,19,1,0,3,0.9565,0.9500,1.0000,0.9744
OVERALL,30,1,2,13,0.9348,0.9677,0.9375,0.9524
